# **[프로젝트] CAM을 만들고 평가해보자**

**프로젝트의 목표**

*   CAM과 Grad-CAM 각각에 대해 원본이미지합성, 바운딩박스, IoU 계산 과정을 통해 CAM과 Grad-CAM의 object localization 성능을 IoU 지표로 비교분석
*   CAM 방식과 Grad-CAM 방식의 class activation map이 정상적으로 얻어지는지 여부와 시각화하였을 때 해당 object의 주요 특징 위치를 잘 반영하는지를 확인
*   ResNet50 + GAP + DenseLayer 결합된 CAM 모델의 학습과정이 안정적으로 수렴하는 것을 관찰하여 기본 모델의 구성과 학습이 정상인지 여부 확인

**실험 조건**

*   데이터셋
    *   1차 실험 : oxford_iiit_pet(이미지상 가까운 객체가 많은 데이터셋)
    *   2차 실험 : PASCAL VOC 2007(이미지상 멀리 있는 객체가 많은 데이터셋)
*   Epoch : 10(Early stop 적용)
*   성능지료 : IoU(Intersection Over Union)
*   베이스라인 모델 : ResNet50 + GAP + DenseLayer


# **실험 1 : 동물 이미지 객체에 대한 실험**

In [ ]:
# 설치 및 환경설정

!pip -q install tensorflow-datasets opencv-python

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import cv2
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# TFDS 데이터 로드(Oxford-IIIT Pet)

  # species(cat/dog) 라벨로 초보자용 이진분류
  # segmentation_mask로 정답 bbox(GT bbox) 생성

DATASET_NAME = "oxford_iiit_pet"
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

(ds_all, ds_info) = tfds.load(
    DATASET_NAME,
    with_info=True,
    as_supervised=False
)

print(ds_info)
print(ds_all.keys())

In [ ]:
# 전처리(학습용)

def preprocess_for_train(example):
    img = tf.image.resize(example["image"], (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)  # keep 0~255 float for preprocess_input in-model
    label = tf.cast(example["species"], tf.int32)  # 0: cat, 1: dog (dataset-defined)
    return img, label

def preprocess_for_eval(example):
    # eval/시각화용: 원본(image), resize image, label, segmentation
    img0 = example["image"]  # original size 적용
    seg0 = example["segmentation_mask"]
    img = tf.image.resize(img0, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    label = tf.cast(example["species"], tf.int32)
    return img0, seg0, img, label

In [ ]:
# train/val/test 구성

train_raw = ds_all["train"]
test_raw  = ds_all["test"]

# train → train/val split
train_size = ds_info.splits["train"].num_examples
val_size = int(train_size * 0.1)

train_ds = train_raw.skip(val_size).map(preprocess_for_train, num_parallel_calls=AUTOTUNE)
val_ds   = train_raw.take(val_size).map(preprocess_for_train, num_parallel_calls=AUTOTUNE)
test_vis_ds = test_raw.map(preprocess_for_eval, num_parallel_calls=AUTOTUNE)

train_ds = train_ds.shuffle(2048).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds   = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
# 기본모델 : ResNet50 + GAP + Dense

from tensorflow.keras import layers as L

def build_cam_model(num_classes=2):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")

    # ResNet50 expects preprocess_input on 0~255
    x = tf.keras.applications.resnet50.preprocess_input(inputs)

    base = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base.trainable = False  # 1단계: 고정(안정 수렴)

    feat = base.output                                  # (H, W, C)
    gap  = L.GlobalAveragePooling2D(name="gap")(feat)   # (C,)
    logits = L.Dense(num_classes, activation=None, name="classifier")(gap)

    model = tf.keras.Model(inputs=inputs, outputs=logits, name="resnet50_cam")
    return model

model = build_cam_model(num_classes=2)
model.summary()

In [ ]:
# 학습

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

cbs = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor="val_loss"),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=cbs)

In [ ]:
# Overlay 함수(visualize_cam_on_image())

def visualize_cam_on_image(image_rgb, heatmap, alpha=0.4):
    """
    image_rgb: (H,W,3) uint8 RGB
    heatmap:   (H,W) float32 [0,1]
    return: overlay_rgb (H,W,3) uint8 RGB, heatmap_color_rgb
    """
    if image_rgb.dtype != np.uint8:
        image_rgb = np.clip(image_rgb, 0, 255).astype(np.uint8)

    h, w = image_rgb.shape[:2]
    heatmap_rs = cv2.resize(heatmap, (w, h))
    heatmap_u8 = np.uint8(255 * np.clip(heatmap_rs, 0, 1))

    heatmap_color_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    heatmap_color_rgb = cv2.cvtColor(heatmap_color_bgr, cv2.COLOR_BGR2RGB)

    overlay = np.uint8((1 - alpha) * image_rgb + alpha * heatmap_color_rgb)
    return overlay, heatmap_color_rgb

In [ ]:
# CAM 구현 (Dense 가중치로 feature map 가중합)

LAST_CONV_LAYER = "conv5_block3_out"  # ResNet50 마지막 conv 출력

def get_cam_heatmap(model, img_1, last_conv_layer_name=LAST_CONV_LAYER, class_index=None):
    """
    img_1: (1,224,224,3) float32 (0~255)
    return heatmap: (h,w) float32 [0,1], pred_class, probs
    """
    # conv 출력과 logits를 동시에 얻는 모델
    conv_layer = model.get_layer(last_conv_layer_name)
    cam_model = tf.keras.Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    conv_out, logits = cam_model(img_1, training=False)   # conv_out: (1,h,w,c)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
    pred_class = int(np.argmax(probs)) if class_index is None else int(class_index)

    # Dense(classifier) weights: (C, num_classes)
    W, b = model.get_layer("classifier").get_weights()
    class_w = W[:, pred_class]  # (C,)

    # CAM = sum_c (feature_map[:,:,c] * w_c)
    fmap = conv_out[0]  # (h,w,c)
    cam = tf.tensordot(fmap, class_w, axes=(2, 0))  # (h,w)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy().astype(np.float32), pred_class, probs

In [ ]:
# Grad-CAM 구현(여러 레이어 heatmap 확인 포함)

def get_gradcam_heatmap(model, img_1, last_conv_layer_name, class_index=None):
    """
    img_1: (1,224,224,3) float32 (0~255)
    return heatmap (h,w) float32 [0,1], pred_class, probs
    """
    conv_layer = model.get_layer(last_conv_layer_name)
    grad_model = tf.keras.Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        conv_out, logits = grad_model(img_1, training=False)
        probs_tf = tf.nn.softmax(logits, axis=-1)[0]
        pred = tf.argmax(probs_tf) if class_index is None else tf.constant(class_index)
        loss = logits[:, pred]  # logits 기준

    grads = tape.gradient(loss, conv_out)  # (1,h,w,c)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # (c,)

    conv_out = conv_out[0]  # (h,w,c)
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)  # (h,w)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    probs = probs_tf.numpy()
    pred_class = int(tf.argmax(probs_tf).numpy()) if class_index is None else int(class_index)
    return heatmap.numpy().astype(np.float32), pred_class, probs

In [ ]:
# Grad-CAM 여러 레이어 확인

GRADCAM_LAYERS = [
    "conv2_block3_out",
    "conv3_block4_out",
    "conv4_block6_out",
    "conv5_block3_out",
]

In [ ]:
# bbox 추출 : heatmap → contour → bounding box

def cam_to_bbox(heatmap, orig_h, orig_w, thr=0.5):
    """
    heatmap: (h,w) float32 [0,1]
    return bbox: (x1,y1,x2,y2) in original image coords or None
    """
    hm = cv2.resize(heatmap, (orig_w, orig_h))
    hm_u8 = np.uint8(255 * np.clip(hm, 0, 1))

    # threshold
    _, bw = cv2.threshold(hm_u8, int(255 * thr), 255, cv2.THRESH_BINARY)

    # 작은 노이즈 제거
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    # 가장 큰 영역 선택
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    return (x, y, x + w, y + h)

def draw_bbox(image_rgb, bbox, color=(0,255,0), thickness=2, text=None):
    """
    image_rgb: uint8 RGB
    bbox: (x1,y1,x2,y2)
    """
    img = image_rgb.copy()
    if bbox is None:
        return img
    x1, y1, x2, y2 = bbox
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.rectangle(img_bgr, (x1,y1), (x2,y2), color, thickness)
    if text is not None:
        cv2.putText(img_bgr, text, (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# GT bbox : segmentation_mask에서 정답 박스 생성
  # Oxford-IIIT Pet의 마스크는 보통 trimap 형태가 많으므로 아래처럼 객체+경계를 object로 잡아 bbox를 설정

def gt_bbox_from_segmentation(seg_mask):
    """
    seg_mask: (H,W,1) uint8/int, values may be {1,2,3} etc.
    return bbox (x1,y1,x2,y2) or None
    """
    m = np.squeeze(seg_mask)
    # 일반적으로 3이 background로 취급
    if m.max() <= 3:
        obj = (m != 3) & (m != 0)  # background 제외(0도 배경 취급)
    else:
        obj = (m > 0)

    ys, xs = np.where(obj)
    if len(xs) == 0:
        return None
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    return (int(x1), int(y1), int(x2), int(y2))

In [ ]:
# IoU 함수(get_iou())

def get_iou(boxA, boxB):
    """
    box: (x1,y1,x2,y2)
    return IoU in [0,1]
    """
    if boxA is None or boxB is None:
        return 0.0

    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter + 1e-8
    return float(inter / union)

In [ ]:
# 시각화 및 IoU 비교 실행 (CAM과 Grad-CAM)

  # CAM overlay + bbox, Grad-CAM overlay + bbox, GT bbox, IoU를 수치로 출력합니다.
  # Grad-CAM 레이어별 heatmap 확인

def show_one_example(example, thr=0.5, alpha=0.45):
    img0, seg0, img_rs, label = example
    img0 = img0.numpy()
    seg0 = seg0.numpy()
    img_rs = img_rs.numpy()

    # 원본 크기
    H, W = img0.shape[:2]

    # 모델 입력 배치
    img_1 = np.expand_dims(img_rs, axis=0).astype(np.float32)  # (1,224,224,3)

    # GT bbox
    gt_box = gt_bbox_from_segmentation(seg0)

    # CAM
    cam_hm, pred_class, probs = get_cam_heatmap(model, img_1, LAST_CONV_LAYER)
    cam_box = cam_to_bbox(cam_hm, H, W, thr=thr)
    cam_overlay, _ = visualize_cam_on_image(img0, cam_hm, alpha=alpha)

    # Grad-CAM (last layer)
    gcam_hm, _, _ = get_gradcam_heatmap(model, img_1, LAST_CONV_LAYER, class_index=pred_class)
    gcam_box = cam_to_bbox(gcam_hm, H, W, thr=thr)
    gcam_overlay, _ = visualize_cam_on_image(img0, gcam_hm, alpha=alpha)

    # IoU
    iou_cam = get_iou(gt_box, cam_box)
    iou_gcam = get_iou(gt_box, gcam_box)

    # bbox draw
    img_gt = draw_bbox(img0, gt_box, color=(0,255,0), text="GT")
    img_cam = draw_bbox(cam_overlay, cam_box, color=(255,255,0), text=f"CAM IoU={iou_cam:.3f}")
    img_gcam = draw_bbox(gcam_overlay, gcam_box, color=(255,0,255), text=f"Grad-CAM IoU={iou_gcam:.3f}")

    # Grad-CAM multiple layers
    layer_overlays = []
    for ln in GRADCAM_LAYERS:
        hm_l, _, _ = get_gradcam_heatmap(model, img_1, ln, class_index=pred_class)
        ov_l, _ = visualize_cam_on_image(img0, hm_l, alpha=alpha)
        layer_overlays.append((ln, ov_l))

    # plot
    plt.figure(figsize=(16, 10))
    plt.subplot(2, 3, 1); plt.title("Original + GT bbox"); plt.imshow(img_gt); plt.axis("off")
    plt.subplot(2, 3, 2); plt.title("CAM overlay + CAM bbox"); plt.imshow(img_cam); plt.axis("off")
    plt.subplot(2, 3, 3); plt.title("Grad-CAM(last) overlay + bbox"); plt.imshow(img_gcam); plt.axis("off")

    for i, (ln, ov) in enumerate(layer_overlays[:3]):  # 아래 3개만 표시(원하면 늘리세요)
        plt.subplot(2, 3, 4 + i)
        plt.title(f"Grad-CAM @ {ln}")
        plt.imshow(ov); plt.axis("off")

    plt.suptitle(f"True species={int(label.numpy())} | Pred={pred_class} | probs={probs}", y=0.98)
    plt.show()

    return iou_cam, iou_gcam

# 여러 샘플(5장 이상)로 반복 시각화/IoU 확인
N_SHOW_EXP1 = 10
EXP1_THR = 0.5
EXP1_ALPHA = 0.45

cam_list, gcam_list = [], []
for idx, ex in enumerate(test_vis_ds.take(N_SHOW_EXP1), start=1):
    iou_cam, iou_gcam = show_one_example(ex, thr=EXP1_THR, alpha=EXP1_ALPHA)
    cam_list.append(iou_cam)
    gcam_list.append(iou_gcam)
    print(f"[{idx}/{N_SHOW_EXP1}] IoU(CAM)={iou_cam:.4f} | IoU(Grad-CAM)={iou_gcam:.4f}")

print("--- Summary over shown samples ---")
print(f"Mean IoU(CAM)      = {float(np.mean(cam_list)):.4f}")
print(f"Mean IoU(Grad-CAM) = {float(np.mean(gcam_list)):.4f}")

In [ ]:
# 테스트셋에서 평균 IoU 비교(성능 비교)

def evaluate_iou(test_vis_ds, n=200, thr=0.5):
    cam_ious = []
    gcam_ious = []

    for ex in test_vis_ds.take(n):
        img0, seg0, img_rs, _ = ex
        img0 = img0.numpy()
        seg0 = seg0.numpy()
        img_rs = img_rs.numpy()

        H, W = img0.shape[:2]
        img_1 = np.expand_dims(img_rs, axis=0).astype(np.float32)

        gt_box = gt_bbox_from_segmentation(seg0)

        cam_hm, pred_class, _ = get_cam_heatmap(model, img_1, LAST_CONV_LAYER)
        cam_box = cam_to_bbox(cam_hm, H, W, thr=thr)
        cam_ious.append(get_iou(gt_box, cam_box))

        gcam_hm, _, _ = get_gradcam_heatmap(model, img_1, LAST_CONV_LAYER, class_index=pred_class)
        gcam_box = cam_to_bbox(gcam_hm, H, W, thr=thr)
        gcam_ious.append(get_iou(gt_box, gcam_box))

    return float(np.mean(cam_ious)), float(np.mean(gcam_ious))

mean_cam, mean_gcam = evaluate_iou(test_vis_ds, n=200, thr=0.5)
print(f"Mean IoU over 200 samples | CAM={mean_cam:.4f} | Grad-CAM={mean_gcam:.4f}")

In [ ]:
# threshold의 변경을 통한 CAM와 Grad-CAM 비교

for thr in [0.3, 0.4, 0.5, 0.6]:
    mean_cam, mean_gcam = evaluate_iou(test_vis_ds, n=200, thr=thr)
    print(f"thr={thr:.1f} | CAM={mean_cam:.4f} | Grad-CAM={mean_gcam:.4f}")

# **실험 2 : 동물 이외 객체에 대한 실험**

In [ ]:
# 실험 2 설정

!pip -q install tensorflow-datasets opencv-python

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import cv2
import matplotlib.pyplot as plt

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

DATASET_NAME = "voc/2007"

# 유형 정의 (bbox 면적 비율 기준: normalized area)
# min_area/max_area는 "정규화 면적" (0~1)
SAMPLE_TYPES = {
    "train_default": dict(min_area=0.003, max_area=0.20,  min_center_dist=0.0, require_multi=False),
    "tiny":         dict(min_area=0.001, max_area=0.010, min_center_dist=0.0, require_multi=False),
    "small":        dict(min_area=0.010, max_area=0.040, min_center_dist=0.0, require_multi=False),
    "medium":       dict(min_area=0.040, max_area=0.120, min_center_dist=0.0, require_multi=False),
    "large":        dict(min_area=0.120, max_area=0.60,  min_center_dist=0.0, require_multi=False),
    # 중심에서 먼(가장자리) + 작은 객체
    "edge_small":   dict(min_area=0.005, max_area=0.050, min_center_dist=0.35, require_multi=False),
    # 다중 객체 이미지에서 작은 객체
    "multi_small":  dict(min_area=0.005, max_area=0.050, min_center_dist=0.0, require_multi=True),
}

# 실험 컨트롤
TRAIN_TYPE = "train_default"
EVAL_TYPES = ["tiny", "small", "medium", "large", "edge_small", "multi_small"]

N_EVAL_PER_TYPE = 200       # 유형별 평균 IoU 계산에 사용할 샘플 수
N_SHOW_PER_TYPE = 10        # 유형별 시각화 샘플 수
HEATMAP_THR = 0.5
OVERLAY_ALPHA = 0.45


In [ ]:
# 데이터셋 로드 및 클래스 정의

(ds_all, ds_info) = tfds.load(DATASET_NAME, with_info=True, as_supervised=False)
print(ds_all.keys())  # train, validation, test

class_names = ds_info.features["objects"]["label"].names
NUM_CLASSES = len(class_names)
print("NUM_CLASSES:", NUM_CLASSES)
print("classes:", class_names)


In [ ]:
# 유형별 객체 선택 로직
# VOC는 이미지 1장에 객체가 여러 개이므로, 유형 조건에 맞는 객체 1개를 골라서:
# - label = 그 객체의 class
# - GT bbox = 그 객체의 bbox
# 로 고정해 실험합니다.

def _bbox_area_ratio(bboxes):
    # bboxes: (N,4) normalized [ymin,xmin,ymax,xmax]
    y1, x1, y2, x2 = bboxes[:,0], bboxes[:,1], bboxes[:,2], bboxes[:,3]
    return tf.maximum(0.0, (y2 - y1)) * tf.maximum(0.0, (x2 - x1))

def _bbox_center_dist_from_image_center(bboxes):
    # normalized 좌표에서 center=(0.5,0.5)까지의 유클리드 거리
    y1, x1, y2, x2 = bboxes[:,0], bboxes[:,1], bboxes[:,2], bboxes[:,3]
    cy = (y1 + y2) * 0.5
    cx = (x1 + x2) * 0.5
    return tf.sqrt(tf.square(cx - 0.5) + tf.square(cy - 0.5))

def select_object_index_by_type(bboxes, sample_type):
    """bboxes: (N,4) normalized. return (idx:int32, valid:bool)."""
    cfg = SAMPLE_TYPES[sample_type]
    min_area = tf.constant(cfg["min_area"], tf.float32)
    max_area = tf.constant(cfg["max_area"], tf.float32)
    min_center_dist = tf.constant(cfg["min_center_dist"], tf.float32)
    require_multi = tf.constant(cfg["require_multi"])

    n = tf.shape(bboxes)[0]
    areas = _bbox_area_ratio(bboxes)
    dists = _bbox_center_dist_from_image_center(bboxes)

    mask = tf.logical_and(areas >= min_area, areas <= max_area)
    mask = tf.logical_and(mask, dists >= min_center_dist)

    if_true = lambda: tf.constant(True)
    if_false = lambda: tf.constant(False)
    multi_ok = tf.cond(n >= 2, if_true, if_false)
    valid = tf.logical_and(tf.reduce_any(mask), tf.logical_or(tf.logical_not(require_multi), multi_ok))

    cand = tf.where(mask)[:,0]

    def pick():
        # 후보 중 "가장 작은 객체"를 선택(멀리 있는 객체에 더 유리)
        cand_areas = tf.gather(areas, cand)
        best = tf.gather(cand, tf.argmin(cand_areas))
        return tf.cast(best, tf.int32), tf.constant(True)

    def none():
        return tf.constant(-1, tf.int32), tf.constant(False)

    return tf.cond(valid, pick, none)


In [ ]:
# 전처리(유형별): 학습용 (image_resized, label) + valid

def preprocess_for_train_by_type(example, sample_type):
    img0 = example["image"]  # uint8
    bboxes = tf.cast(example["objects"]["bbox"], tf.float32)
    labels = tf.cast(example["objects"]["label"], tf.int32)

    idx, valid = select_object_index_by_type(bboxes, sample_type)
    label = tf.cond(valid, lambda: tf.gather(labels, idx), lambda: tf.constant(0, tf.int32))

    img = tf.image.resize(img0, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)  # 0~255

    return img, label, valid

# 전처리(유형별): 평가/시각화용 (orig_img, gt_bbox_pixel, resized_img, label) + valid
def preprocess_for_eval_by_type(example, sample_type):
    img0 = example["image"]
    H = tf.shape(img0)[0]
    W = tf.shape(img0)[1]

    bboxes = tf.cast(example["objects"]["bbox"], tf.float32)
    labels = tf.cast(example["objects"]["label"], tf.int32)

    idx, valid = select_object_index_by_type(bboxes, sample_type)
    bbox = tf.cond(valid, lambda: tf.gather(bboxes, idx), lambda: tf.zeros([4], tf.float32))
    label = tf.cond(valid, lambda: tf.gather(labels, idx), lambda: tf.constant(0, tf.int32))

    # normalized -> pixel bbox (x1,y1,x2,y2)
    y1, x1, y2, x2 = bbox[0], bbox[1], bbox[2], bbox[3]
    x1p = tf.cast(x1 * tf.cast(W, tf.float32), tf.int32)
    x2p = tf.cast(x2 * tf.cast(W, tf.float32), tf.int32)
    y1p = tf.cast(y1 * tf.cast(H, tf.float32), tf.int32)
    y2p = tf.cast(y2 * tf.cast(H, tf.float32), tf.int32)
    gt_box = tf.stack([x1p, y1p, x2p, y2p], axis=0)

    img = tf.image.resize(img0, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)

    return img0, gt_box, img, label, valid


In [ ]:
# 유형별 데이터셋 생성 함수

train_raw = ds_all["train"]
val_raw   = ds_all["validation"]
test_raw  = ds_all["test"]

def make_train_ds(sample_type):
    return (train_raw
            .map(lambda ex: preprocess_for_train_by_type(ex, sample_type), num_parallel_calls=AUTOTUNE)
            .filter(lambda img, label, valid: valid)
            .map(lambda img, label, valid: (img, label))
            .shuffle(2048)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

def make_val_ds(sample_type):
    return (val_raw
            .map(lambda ex: preprocess_for_train_by_type(ex, sample_type), num_parallel_calls=AUTOTUNE)
            .filter(lambda img, label, valid: valid)
            .map(lambda img, label, valid: (img, label))
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

def make_test_vis_ds(sample_type):
    return (test_raw
            .map(lambda ex: preprocess_for_eval_by_type(ex, sample_type), num_parallel_calls=AUTOTUNE)
            .filter(lambda img0, gt, img, label, valid: valid)
            .map(lambda img0, gt, img, label, valid: (img0, gt, img, label))
            .prefetch(AUTOTUNE))

train_ds = make_train_ds(TRAIN_TYPE)
val_ds   = make_val_ds(TRAIN_TYPE)

In [ ]:
# 모델: ResNet50 + GAP + Dense (CAM 가능 구조)

from tensorflow.keras import layers as L

LAST_CONV_LAYER = "conv5_block3_out"

def build_cam_model(num_classes):
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")
    x = tf.keras.applications.resnet50.preprocess_input(inputs)  # expects 0~255
    base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=x)
    base.trainable = False

    feat = base.output
    gap = L.GlobalAveragePooling2D(name="gap")(feat)
    logits = L.Dense(num_classes, activation=None, name="classifier")(gap)
    model = tf.keras.Model(inputs, logits, name="resnet50_cam")
    return model

model = build_cam_model(NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
model.summary()

cbs = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5, monitor="val_loss"),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=cbs)

In [ ]:
# CAM/Grad-CAM 시각화 + bbox + IoU 유틸

def visualize_cam_on_image(image_rgb, heatmap, alpha=0.4):
    """원본 이미지와 heatmap을 alpha로 섞어 overlay 반환"""
    if image_rgb.dtype != np.uint8:
        image_rgb = np.clip(image_rgb, 0, 255).astype(np.uint8)

    h, w = image_rgb.shape[:2]
    heatmap_rs = cv2.resize(heatmap, (w, h))
    heatmap_u8 = np.uint8(255 * np.clip(heatmap_rs, 0, 1))

    heatmap_color_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    heatmap_color_rgb = cv2.cvtColor(heatmap_color_bgr, cv2.COLOR_BGR2RGB)

    overlay = np.uint8((1 - alpha) * image_rgb + alpha * heatmap_color_rgb)
    return overlay, heatmap_color_rgb

def get_cam_heatmap(model, img_1, last_conv_layer_name=LAST_CONV_LAYER, class_index=None):
    conv_layer = model.get_layer(last_conv_layer_name)
    cam_model = tf.keras.Model(model.inputs, [conv_layer.output, model.output])

    conv_out, logits = cam_model(img_1, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
    pred_class = int(np.argmax(probs)) if class_index is None else int(class_index)

    W, _b = model.get_layer("classifier").get_weights()  # (C, num_classes)
    class_w = W[:, pred_class]  # (C,)

    fmap = conv_out[0]  # (h,w,c)
    cam = tf.tensordot(fmap, class_w, axes=(2, 0))  # (h,w)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy().astype(np.float32), pred_class, probs

def get_gradcam_heatmap(model, img_1, last_conv_layer_name, class_index=None):
    conv_layer = model.get_layer(last_conv_layer_name)
    grad_model = tf.keras.Model(model.inputs, [conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        conv_out, logits = grad_model(img_1, training=False)
        probs_tf = tf.nn.softmax(logits, axis=-1)[0]
        pred = tf.argmax(probs_tf) if class_index is None else tf.constant(class_index)
        loss = logits[:, pred]

    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # (c,)

    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    pred_class = int(tf.argmax(probs_tf).numpy()) if class_index is None else int(class_index)
    return heatmap.numpy().astype(np.float32), pred_class, probs_tf.numpy()

def cam_to_bbox(heatmap, orig_h, orig_w, thr=0.5):
    hm = cv2.resize(heatmap, (orig_w, orig_h))
    hm_u8 = np.uint8(255 * np.clip(hm, 0, 1))
    _, bw = cv2.threshold(hm_u8, int(255 * thr), 255, cv2.THRESH_BINARY)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))

    contours, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    return (x, y, x+w, y+h)

def draw_bbox(image_rgb, bbox, color=(0,255,0), thickness=2, text=None):
    img = image_rgb.copy()
    if bbox is None:
        return img
    x1, y1, x2, y2 = bbox
    bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.rectangle(bgr, (x1,y1), (x2,y2), color, thickness)
    if text is not None:
        cv2.putText(bgr, text, (x1, max(0,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def get_iou(boxA, boxB):
    if boxA is None or boxB is None:
        return 0.0
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    return float(inter / (areaA + areaB - inter + 1e-8))


In [ ]:
# 유형별로 여러 장 시각화 + 유형별 평균 IoU 비교

GRADCAM_LAYERS = [
    "conv2_block3_out",
    "conv3_block4_out",
    "conv4_block6_out",
    "conv5_block3_out",
]

def show_one_example(img0, gt_box, img_rs, true_label, thr=0.5, alpha=0.45):
    img0 = np.array(img0)
    gt_box = tuple(np.array(gt_box).tolist())
    img_rs = np.array(img_rs)

    H, W = img0.shape[:2]
    img_1 = np.expand_dims(img_rs, axis=0).astype(np.float32)

    cam_hm, pred_class, probs = get_cam_heatmap(model, img_1, LAST_CONV_LAYER)
    cam_box = cam_to_bbox(cam_hm, H, W, thr=thr)
    cam_overlay, _ = visualize_cam_on_image(img0, cam_hm, alpha=alpha)

    gcam_hm, _, _ = get_gradcam_heatmap(model, img_1, LAST_CONV_LAYER, class_index=pred_class)
    gcam_box = cam_to_bbox(gcam_hm, H, W, thr=thr)
    gcam_overlay, _ = visualize_cam_on_image(img0, gcam_hm, alpha=alpha)

    iou_cam  = get_iou(gt_box, cam_box)
    iou_gcam = get_iou(gt_box, gcam_box)

    img_gt  = draw_bbox(img0, gt_box, color=(0,255,0), text="GT")
    img_cam = draw_bbox(cam_overlay, cam_box, color=(255,255,0), text=f"CAM IoU={iou_cam:.3f}")
    img_gcm = draw_bbox(gcam_overlay, gcam_box, color=(255,0,255), text=f"Grad-CAM IoU={iou_gcam:.3f}")

    # 레이어별 Grad-CAM
    layer_imgs = []
    for ln in GRADCAM_LAYERS:
        hm_l, _, _ = get_gradcam_heatmap(model, img_1, ln, class_index=pred_class)
        ov_l, _ = visualize_cam_on_image(img0, hm_l, alpha=alpha)
        layer_imgs.append((ln, ov_l))

    plt.figure(figsize=(18, 10))
    plt.subplot(2,3,1); plt.title("Original + GT bbox"); plt.imshow(img_gt); plt.axis("off")
    plt.subplot(2,3,2); plt.title("CAM overlay + bbox"); plt.imshow(img_cam); plt.axis("off")
    plt.subplot(2,3,3); plt.title("Grad-CAM(last) overlay + bbox"); plt.imshow(img_gcm); plt.axis("off")

    for i, (ln, ov) in enumerate(layer_imgs[:3]):
        plt.subplot(2,3,4+i); plt.title(f"Grad-CAM @ {ln}"); plt.imshow(ov); plt.axis("off")

    true_name = class_names[int(true_label)]
    pred_name = class_names[int(pred_class)]
    plt.suptitle(f"True={true_name} | Pred={pred_name} | IoU(CAM)={iou_cam:.3f}, IoU(Grad-CAM)={iou_gcam:.3f}", y=0.98)
    plt.show()

    return iou_cam, iou_gcam

def show_examples_for_type(sample_type, k=5, thr=0.5, alpha=0.45):
    ds = make_test_vis_ds(sample_type)
    cam_ious, gcam_ious = [], []
    for i, (img0, gt_box, img_rs, label) in enumerate(ds.take(k)):
        iou_cam, iou_gcam = show_one_example(img0.numpy(), gt_box.numpy(), img_rs.numpy(), label.numpy(),
                                             thr=thr, alpha=alpha)
        cam_ious.append(iou_cam)
        gcam_ious.append(iou_gcam)
    return cam_ious, gcam_ious

def evaluate_iou(ds, n=200, thr=0.5):
    cam_ious, gcam_ious = [], []
    for img0, gt_box, img_rs, _label in ds.take(n):
        img0 = img0.numpy()
        gt_box = tuple(gt_box.numpy().tolist())
        img_rs = img_rs.numpy()
        H, W = img0.shape[:2]
        img_1 = np.expand_dims(img_rs, axis=0).astype(np.float32)

        cam_hm, pred_class, _ = get_cam_heatmap(model, img_1, LAST_CONV_LAYER)
        cam_box = cam_to_bbox(cam_hm, H, W, thr=thr)
        cam_ious.append(get_iou(gt_box, cam_box))

        gcam_hm, _, _ = get_gradcam_heatmap(model, img_1, LAST_CONV_LAYER, class_index=pred_class)
        gcam_box = cam_to_bbox(gcam_hm, H, W, thr=thr)
        gcam_ious.append(get_iou(gt_box, gcam_box))

    return float(np.mean(cam_ious)), float(np.mean(gcam_ious))

def evaluate_iou_by_type(eval_types, n_per_type=200, thr=0.5):
    results = {}
    for t in eval_types:
        ds = make_test_vis_ds(t)
        m_cam, m_gcam = evaluate_iou(ds, n=n_per_type, thr=thr)
        results[t] = (m_cam, m_gcam)
        print(f"[{t:10s}] Mean IoU | CAM={m_cam:.4f} | Grad-CAM={m_gcam:.4f}  (n={n_per_type}, thr={thr})")
    return results


In [ ]:
# 유형별 여러 장 시각화 및 유형별 평균 IoU 비교

print("TRAIN_TYPE =", TRAIN_TYPE)
print("EVAL_TYPES  =", EVAL_TYPES)

print("\n(1) 유형별 예시 시각화")
for t in EVAL_TYPES:
    print(f"\n--- SHOW: {t} (k={N_SHOW_PER_TYPE}) ---")
    show_examples_for_type(t, k=N_SHOW_PER_TYPE, thr=HEATMAP_THR, alpha=OVERLAY_ALPHA)

print("\n(2) 유형별 평균 IoU 비교")
results = evaluate_iou_by_type(EVAL_TYPES, n_per_type=N_EVAL_PER_TYPE, thr=HEATMAP_THR)


객체 유형

*   tiny : 객체가 작게 나온 이미지
*   small : tiny보다 큰 객체로 나온 이미지
*   medium : 중간 크기의 객체로 나온 이미지
*   large : 객체가 차지하는 비율이 큰 이미지
*   edge_small : 화면 가장자리에 작은 객체가 있는 이미지
*   multi_small : 여러 객체가 있는 이미지



**실험 결과**

*   실험 1을 통하여 ResNet50+GAP+Dense이 결합된 CAM 모델의 학습 과정이 안정적으로 수렴하였음을 확인(Epoch 2~6에서 Train acc가 0.99 → 1.00 근처로 상승하고 Val acc는 0.99로 안정된 것으로 판단)
*   실험 2을 통하여 Val acc가 57%대를 나타냈으나, Loss가 감소 추세를 보인다는 점에서 학습 과정이 안정적으로 수렴하였음을 확인
*   실험 1과 2 모두 CAM과 Grad-CAM의 샘플 200개에 대해서 평균 IoU가 모두 동일한 결과 도출


**결과 분석**

*   실험 1에서 CAM과 Grad-CAM의 동물 객체가 배경 이미지가 경계가 뚜렷한 이미지의 경우 오버레이가 객체 중심부를 강조하는 형태로 나타났고, bbox이 heatmap 기반으로 생성되어 설명 가능한 시각화 목적 달성
*   실험 2에서 tiny(객체가 작게 나온 이미지)에서 가장 IoU가 낮은데(0.0088), 작은 객체일 수록 Heatmap을 원본 크기로 키우는 과정에서 경계가 뭉개지는 현상이 일어났으며 bbox가 조금만 틀어져도 IoU가 급락하는 것을 보여줌
*   실험 1에서 추가 실험으로 threshold 변화에 따른 성능 변화가 있을 것이라고 가정한 점을 실험한 결과로 object localization 성능은 threshold에 민감하게 반응함을 확인(thr=0.3에서 평균 IoU가 0.34로 최고값이며, thr를 올릴수록 IoU가 감소)
*   CAM과 Grad-CAM의 IoU가 모든 실험에서 동일한 것은 Conv → GAP → Linear(Dense)의 구조에서 GAP은 feature map을 평균내고, Dense는 그 평균값을 가중치로 더해서 점수를 만드는 구조임. Grad-CAM이 계산하는 중요도값(gradient의 평균)은 Dense의 가중치 계산과 같은 역할을 함(출처 : Empowering CAM-Based Methods with Capability to Generate
Fine-Grained and High-Faithfulness Explanations(https://arxiv.org/pdf/2303.09171))





**한계점 및 보완점**


*   CAM과 Grad-CAM의 IoU 결과가 동일한데 Grad-CAM의 타깃 레이어를 마지막(conv5) 말고 중간 레이어(conv3이나 4)로 변경하여 비교한다면 달라질 수도 있을 것이라 판단됨
*   Early Stopping을 적용하였지만, localization 성능을 올리기 위한 방안(미세조정, 증강, 클래스 균형 등)은 적용하지 않았으므로 추후 성능 향상 방법 모색



**회고**

*   IoU가 같다고 하여 CAM과 Grad-CAM이 같은 모델이라고 결론을 내리는 것보다 IoU를 계산하는 구조가 동일하기 때문에 계산 방식을 변화시키면 달라질 수 있다는 사실을 알게 되었음
*   IoU는 위치 매트릭로 얼마나 정확히 같은 영역을 덮었는지를 나타내는 직관적인 지표로 판단됨
*   베이스라인 모델로 ResNet50 + GAP + DenseLayer말고도 타 모델 적용으로도 안정적으로 수렴 가능한지 더 실험할 수 있을 것 같음
